 imports libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import cv2
import os
import json
import math
from PIL import Image
from torch.autograd import Variable
from torch.utils.data import Subset
from torchvision.datasets import CocoDetection
from typing import Any, Callable, Optional, Tuple
import albumentations as A
import darknet
from util import *

print("All imports done!")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

All imports done!
PyTorch version: 2.5.1+cu121
CUDA available: True
GPU count: 4


paths and constants:

In [3]:
# Paths
path2data = '/home/jupyter-st125985/fiftyone/coco-2017/validation/data/val2017'
path2json = '/home/jupyter-st125985/fiftyone/coco-2017/raw/annotations/instances_val2017.json'

# Constants
img_size = 416
NUM_CLASSES = 80
NUM_ANCHORS = 3
BATCH_SIZE = 4

ANCHORS = [
    [[12, 16], [19, 36], [40, 28]],
    [[36, 75], [76, 55], [72, 146]],
    [[142, 110], [192, 243], [459, 401]]
]

STRIDES = [8, 16, 32]

# Device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

print("Device:", device)
print("Image size:", img_size)
print("Batch size:", BATCH_SIZE)

Device: cuda:0
Image size: 416
Batch size: 4


load the COCO categories:

In [4]:
# Load COCO categories
with open('coco_cats.json') as js:
    data = json.load(js)["categories"]

cats_dict = {}
for i in range(80):
    cats_dict[str(data[i]['id'])] = i

print("Categories loaded!")
print("Sample:", list(cats_dict.items())[:5])

Categories loaded!
Sample: [('1', 0), ('2', 1), ('3', 2), ('4', 3), ('5', 4)]


helper functions:

In [5]:
def iou_xywh_numpy(boxes1, boxes2):
    boxes1 = np.array(boxes1)
    boxes2 = np.array(boxes2)

    boxes1_area = boxes1[..., 2] * boxes1[..., 3]
    boxes2_area = boxes2[..., 2] * boxes2[..., 3]

    boxes1 = np.concatenate([boxes1[..., :2] - boxes1[..., 2:] * 0.5,
                              boxes1[..., :2] + boxes1[..., 2:] * 0.5], axis=-1)
    boxes2 = np.concatenate([boxes2[..., :2] - boxes2[..., 2:] * 0.5,
                              boxes2[..., :2] + boxes2[..., 2:] * 0.5], axis=-1)

    left_up = np.maximum(boxes1[..., :2], boxes2[..., :2])
    right_down = np.minimum(boxes1[..., 2:], boxes2[..., 2:])

    inter_section = np.maximum(right_down - left_up, 0.0)
    inter_area = inter_section[..., 0] * inter_section[..., 1]
    union_area = boxes1_area + boxes2_area - inter_area
    IOU = 1.0 * inter_area / (union_area + 1e-6)
    return IOU


def CIOU_xywh_torch(boxes1, boxes2):
    boxes1 = torch.cat([boxes1[..., :2] - boxes1[..., 2:] * 0.5,
                        boxes1[..., :2] + boxes1[..., 2:] * 0.5], dim=-1)
    boxes2 = torch.cat([boxes2[..., :2] - boxes2[..., 2:] * 0.5,
                        boxes2[..., :2] + boxes2[..., 2:] * 0.5], dim=-1)

    boxes1_area = (boxes1[..., 2] - boxes1[..., 0]) * (boxes1[..., 3] - boxes1[..., 1])
    boxes2_area = (boxes2[..., 2] - boxes2[..., 0]) * (boxes2[..., 3] - boxes2[..., 1])

    inter_left_up = torch.max(boxes1[..., :2], boxes2[..., :2])
    inter_right_down = torch.min(boxes1[..., 2:], boxes2[..., 2:])
    inter_section = torch.max(inter_right_down - inter_left_up, torch.zeros_like(inter_right_down))
    inter_area = inter_section[..., 0] * inter_section[..., 1]
    union_area = boxes1_area + boxes2_area - inter_area
    ious = 1.0 * inter_area / (union_area + 1e-6)

    outer_left_up = torch.min(boxes1[..., :2], boxes2[..., :2])
    outer_right_down = torch.max(boxes1[..., 2:], boxes2[..., 2:])
    outer = torch.max(outer_right_down - outer_left_up, torch.zeros_like(inter_right_down))
    outer_diagonal_line = torch.pow(outer[..., 0], 2) + torch.pow(outer[..., 1], 2)

    boxes1_center = (boxes1[..., :2] + boxes1[..., 2:]) * 0.5
    boxes2_center = (boxes2[..., :2] + boxes2[..., 2:]) * 0.5
    center_dis = torch.pow(boxes1_center[..., 0] - boxes2_center[..., 0], 2) + \
                 torch.pow(boxes1_center[..., 1] - boxes2_center[..., 1], 2)

    boxes1_size = torch.max(boxes1[..., 2:] - boxes1[..., :2], torch.zeros_like(inter_right_down))
    boxes2_size = torch.max(boxes2[..., 2:] - boxes2[..., :2], torch.zeros_like(inter_right_down))
    v = (4 / (math.pi ** 2)) * torch.pow(
        torch.atan(boxes1_size[..., 0] / torch.clamp(boxes1_size[..., 1], min=1e-6)) -
        torch.atan(boxes2_size[..., 0] / torch.clamp(boxes2_size[..., 1], min=1e-6)), 2)

    alpha = v / (1 - ious + v + 1e-6)
    cious = ious - (center_dis / (outer_diagonal_line + 1e-6) + alpha * v)
    return cious

print("Helper functions defined!")

Helper functions defined!


add the CustomCoco class:

In [6]:
class CustomCoco(CocoDetection):
    def __init__(self, root, annFile, transform=None, target_transform=None, transforms=None):
        super(CocoDetection, self).__init__(root, transforms, transform, target_transform)
        from pycocotools.coco import COCO
        self.coco = COCO(annFile)
        self.ids = list(sorted(self.coco.imgs.keys()))

    def __getitem__(self, index):
        coco = self.coco
        img_id = self.ids[index]
        ann_ids = coco.getAnnIds(imgIds=img_id)
        target = coco.loadAnns(ann_ids)
        path = coco.loadImgs(img_id)[0]['file_name']

        img = Image.open(os.path.join(self.root, path)).convert('RGB')
        img = np.array(img)

        category_ids = list(obj['category_id'] for obj in target)
        bboxes = list(obj['bbox'] for obj in target)

        if self.transform is not None:
            transformed = self.transform(image=img, bboxes=bboxes, category_ids=category_ids)
            img = transformed['image']
            bboxes = torch.Tensor(transformed['bboxes'])
            cat_ids = torch.Tensor(transformed['category_ids'])
            labels, bboxes = self.__create_label(bboxes, cat_ids.type(torch.IntTensor))

        return img, labels, bboxes

    def __len__(self):
        return len(self.ids)

    def __create_label(self, bboxes, class_inds):
        bboxes = np.array(bboxes)
        class_inds = np.array(class_inds)
        strides = np.array(STRIDES)
        train_output_size = img_size / strides
        anchors_per_scale = NUM_ANCHORS

        label = [
            np.zeros((int(train_output_size[i]), int(train_output_size[i]), anchors_per_scale, 5 + NUM_CLASSES))
            for i in range(3)
        ]
        bboxes_xywh = [np.zeros((150, 4)) for _ in range(3)]
        bbox_count = np.zeros((3,))

        for i in range(len(bboxes)):
            bbox_coor = bboxes[i][:4]
            bbox_class_ind = cats_dict[str(class_inds[i])]

            one_hot = np.zeros(NUM_CLASSES, dtype=np.float32)
            one_hot[bbox_class_ind] = 1.0

            bbox_xywh = np.concatenate([
                (0.5 * bbox_coor[2:] + bbox_coor[:2]),
                bbox_coor[2:]
            ], axis=-1)

            bbox_xywh_scaled = 1.0 * bbox_xywh[np.newaxis, :] / strides[:, np.newaxis]

            iou = []
            exist_positive = False
            for i in range(3):
                anchors_xywh = np.zeros((anchors_per_scale, 4))
                anchors_xywh[:, 0:2] = np.floor(bbox_xywh_scaled[i, 0:2]).astype(np.int32) + 0.5
                anchors_xywh[:, 2:4] = ANCHORS[i]

                iou_scale = iou_xywh_numpy(bbox_xywh_scaled[i][np.newaxis, :], anchors_xywh)
                iou.append(iou_scale)
                iou_mask = iou_scale > 0.3

                if np.any(iou_mask):
                    xind, yind = np.floor(bbox_xywh_scaled[i, 0:2]).astype(np.int32)
                    label[i][yind, xind, iou_mask, 0:4] = bbox_xywh * strides[i]
                    label[i][yind, xind, iou_mask, 4:5] = 1.0
                    label[i][yind, xind, iou_mask, 5:] = one_hot

                    bbox_ind = int(bbox_count[i] % 150)
                    bboxes_xywh[i][bbox_ind, :4] = bbox_xywh * strides[i]
                    bbox_count[i] += 1
                    exist_positive = True

            if not exist_positive:
                best_anchor_ind = np.argmax(np.array(iou).reshape(-1), axis=-1)
                best_detect = int(best_anchor_ind / anchors_per_scale)
                best_anchor = int(best_anchor_ind % anchors_per_scale)
                xind, yind = np.floor(bbox_xywh_scaled[best_detect, 0:2]).astype(np.int32)

                label[best_detect][yind, xind, best_anchor, 0:4] = bbox_xywh * strides[best_detect]
                label[best_detect][yind, xind, best_anchor, 4:5] = 1.0
                label[best_detect][yind, xind, best_anchor, 5:] = one_hot

                bbox_ind = int(bbox_count[best_detect] % 150)
                bboxes_xywh[best_detect][bbox_ind, :4] = bbox_xywh * strides[best_detect]
                bbox_count[best_detect] += 1

        flatten_size_s = int(train_output_size[2]) * int(train_output_size[2]) * anchors_per_scale
        flatten_size_m = int(train_output_size[1]) * int(train_output_size[1]) * anchors_per_scale
        flatten_size_l = int(train_output_size[0]) * int(train_output_size[0]) * anchors_per_scale

        label_s = torch.Tensor(label[2]).view(1, flatten_size_s, 5 + NUM_CLASSES).squeeze(0)
        label_m = torch.Tensor(label[1]).view(1, flatten_size_m, 5 + NUM_CLASSES).squeeze(0)
        label_l = torch.Tensor(label[0]).view(1, flatten_size_l, 5 + NUM_CLASSES).squeeze(0)

        labels = torch.cat([label_l, label_m, label_s], 0)
        bboxes = torch.cat([torch.Tensor(bboxes_xywh[0]), torch.Tensor(bboxes_xywh[1]), torch.Tensor(bboxes_xywh[2])], 0)
        return labels, bboxes

print("CustomCoco class defined!")

CustomCoco class defined!


create the dataloader:

In [7]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_transform = A.Compose([
    A.Resize(img_size, img_size),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']))

# Use 500 images
train_dataset = Subset(
    CustomCoco(root=path2data, annFile=path2json, transform=train_transform),
    list(range(0, 500))
)

train_dataloader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    collate_fn=collate_fn
)

print("Dataloader ready!")
print("Total images:", len(train_dataset))
print("Total batches:", len(train_dataloader))

loading annotations into memory...
Done (t=0.48s)
creating index...
index created!
Dataloader ready!
Total images: 500
Total batches: 125


load the model:

In [8]:
# Load YOLOv4 model with pretrained weights
model = darknet.Darknet("cfg/yolov4.cfg")
model.load_weights("yolov4.weights")
model.net_info["height"] = str(img_size)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Model loaded!")
print("Device:", device)

Model loaded!
Device: cuda:0


add the train_yolo() function:

In [9]:
def train_yolo(model, optimizer, dataloader, device, img_size, n_epoch, use_ciou=False):
    model.train()
    loss_history = []
    
    for epoch in range(n_epoch):
        running_loss = 0.0
        for batch_idx, (inputs, labels, bboxes) in enumerate(dataloader):
            inputs = torch.from_numpy(np.array(inputs)).squeeze(1).permute(0,3,1,2).float()
            inputs = inputs.to(device)
            labels = torch.stack(labels).to(device)

            optimizer.zero_grad()

            outputs = model(inputs, torch.cuda.is_available())

            pred_xywh = outputs[..., 0:4] / img_size
            pred_conf = outputs[..., 4:5]
            pred_cls  = outputs[..., 5:]

            label_xywh     = labels[..., :4] / img_size
            label_obj_mask = labels[..., 4:5]
            label_noobj    = 1.0 - label_obj_mask
            label_cls      = labels[..., 5:]

            bce = nn.BCELoss(reduction='none')

            if use_ciou:
                ciou = CIOU_xywh_torch(pred_xywh, label_xywh).unsqueeze(-1)
                loss_box = torch.sum(label_obj_mask * (1.0 - ciou))
            else:
                mse = nn.MSELoss(reduction='none')
                loss_box = torch.sum(label_obj_mask * mse(pred_xywh, label_xywh))

            loss_conf = torch.sum(
                label_obj_mask * bce(pred_conf, label_obj_mask) +
                0.05 * label_noobj * bce(pred_conf, label_obj_mask)
            )
            loss_cls = torch.sum(label_obj_mask * bce(pred_cls, label_cls))

            loss = loss_box + loss_conf + loss_cls
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f"  Epoch {epoch+1}, Batch {batch_idx}/{len(dataloader)}, Loss: {loss.item():.4f}")

        epoch_loss = running_loss / len(dataloader)
        loss_history.append(epoch_loss)
        print(f">>> Epoch {epoch+1}/{n_epoch}  Average Loss: {epoch_loss:.4f}")
    
    return loss_history

print("train_yolo() defined!")

train_yolo() defined!


training with MSE loss:

In [10]:
print("Starting training with MSE loss...")
loss_history_mse = train_yolo(
    model, optimizer, train_dataloader,
    device, img_size=img_size, n_epoch=5, use_ciou=False
)

print("\nMSE Training complete!")
print("Loss history:", loss_history_mse)

Starting training with MSE loss...
  Epoch 1, Batch 0/125, Loss: 868.6397
  Epoch 1, Batch 20/125, Loss: 1273.5040
  Epoch 1, Batch 40/125, Loss: 2328.0957
  Epoch 1, Batch 60/125, Loss: 1484.0575
  Epoch 1, Batch 80/125, Loss: 1038.1594
  Epoch 1, Batch 100/125, Loss: 591.2316
  Epoch 1, Batch 120/125, Loss: 1449.3494
>>> Epoch 1/5  Average Loss: nan
  Epoch 2, Batch 0/125, Loss: 2045.9554
  Epoch 2, Batch 20/125, Loss: 748.6590
  Epoch 2, Batch 40/125, Loss: 1524.1849
  Epoch 2, Batch 60/125, Loss: 1539.4991
  Epoch 2, Batch 80/125, Loss: 1483.9819
  Epoch 2, Batch 100/125, Loss: 762.2267
  Epoch 2, Batch 120/125, Loss: 3381.2500
>>> Epoch 2/5  Average Loss: 1014729109725.0797
  Epoch 3, Batch 0/125, Loss: 2242.9282
  Epoch 3, Batch 20/125, Loss: 3999.9680
  Epoch 3, Batch 40/125, Loss: 1607.8572
  Epoch 3, Batch 60/125, Loss: 3085.4666
  Epoch 3, Batch 80/125, Loss: 1709.8054
  Epoch 3, Batch 100/125, Loss: 1027.8524
  Epoch 3, Batch 120/125, Loss: 10901.6377
>>> Epoch 3/5  Average 

In [1]:
import matplotlib.pyplot as plt

# Plot loss history (skip epoch 1 nan)
epochs = [2, 3, 4, 5]
losses = loss_history_mse[1:]

plt.figure(figsize=(8, 5))
plt.plot(epochs, losses, 'b-o', label='MSE Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('YOLOv4 Training Loss (MSE)')
plt.legend()
plt.grid(True)
plt.show()

print("Loss decreasing:", losses[-1] < losses[1])

NameError: name 'loss_history_mse' is not defined